In [3]:
import torch
from torchvision import datasets, models, transforms

torch.manual_seed(42)

transform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset_path= "/content/drive/MyDrive/MBCA/딥러닝 YOLO 심화과정/03 CNN 합성곱 신경망/01 Pytorch/DataSet/Cars"

# 자동차 회사명 분류 CNN 전이학습
첫 번째 셀은 원본 그대로이며, 이미 연결된 Drive와 T4 환경에서 이어서 실행한다. 위 셀의 transform과 dataset_path를 끝까지 그대로 사용한다.

Q1: 차량 정보 인식의 기초를 실습하려고 자동차 이미지를 선택했다. 데이터셋 출처는 [Car Images Dataset](https://www.kaggle.com/datasets/kshitij192/cars-image-dataset)이며, 폴더명을 정답으로 사용한다.
이 파일을 작성한 환경에서는 실제 Drive 데이터와 GPU 학습을 실행하지 않았다. 이후 셀의 이미지 수·학습 로그·성능은 Colab에서 실행해 확인한다.

## 1. ImageFolder로 Train/Test 불러오기
수업 1처럼 하위 폴더명을 라벨로 읽는다. 첫 번째 셀의 전처리와 경로를 재사용하며, train/test는 합치지 않는다.

In [ ]:
dataset= datasets.ImageFolder(root=dataset_path + '/train', transform=transform)
test_dataset= datasets.ImageFolder(root=dataset_path + '/test', transform=transform)
class_names= dataset.classes

# 같은 숫자 라벨이 같은 폴더를 뜻하는지 확인
assert dataset.class_to_idx == test_dataset.class_to_idx, 'Train/Test 클래스 폴더를 확인하세요.'
assert len(class_names) == 7, '7개 클래스 폴더를 확인하세요.'

## 2. 이미지 수와 클래스 분포 확인
파일명이 아닌 폴더명 7개가 정답이다. 클래스별 수를 비교하여 데이터가 한쪽에 치우쳐 있는지 확인한다.

In [ ]:
print('Train 폴더 이미지 수:', len(dataset))
print('Test 폴더 이미지 수:', len(test_dataset))
print('전체 이미지 수:', len(dataset) + len(test_dataset))
print('클래스 수:', len(class_names))
print('클래스명:', class_names)

for i, name in enumerate(class_names):
    train_count= dataset.targets.count(i)
    test_count= test_dataset.targets.count(i)
    print(f'{name}: Train {train_count}장 / Test {test_count}장 / 합계 {train_count + test_count}장')

## 3. 자동차 샘플 확인
수업의 samples와 PIL을 이용하여 각 클래스의 사진을 한 장씩 확인한다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes= plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < len(class_names):
        # 각 클래스에서 첫 사진의 위치를 찾아 원본 이미지 표시
        idx= dataset.targets.index(i)
        image_path, label= dataset.samples[idx]
        image= Image.open(image_path).convert('RGB')
        ax.imshow(image)
        ax.set_title(class_names[label], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Train / Validation / Test 준비
수업 1의 random_split으로 기존 Train의 80%는 학습, 20%는 검증에 사용한다. Test 폴더는 합치거나 나누지 않고 최종 평가에만 사용한다. 난수 시드를 고정해 같은 분할을 사용한다. 무작위 분리이므로 클래스 비율이 정확히 같지는 않을 수 있으며, 원본 폴더 사이에 같은 사진이 없는지도 확인한다.

In [ ]:
from torch.utils.data import random_split

train_dataset, val_dataset= random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)
print('Train:', len(train_dataset))
print('Validation:', len(val_dataset))
print('Test:', len(test_dataset))

## 5. DataLoader 생성
수업과 같은 배치 16을 사용한다. 학습 데이터만 순서를 섞는다.

In [ ]:
from torch.utils.data import DataLoader

train_loader= DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader= DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader= DataLoader(test_dataset, batch_size=16, shuffle=False)

## 6. 사전학습 ResNet50 불러오기 (Q2)
수업의 ResNet152와 같은 방식으로 ResNet50을 사용한다. 잔차 연결을 사용하는 구조를 유지하면서 연산 부담을 줄이기 위한 선택이다.
VGG는 단순한 층 구성이나 파라미터 부담이 크고, MobileNet은 경량 구조에 장점이 있다. 여기서는 수업의 ResNet 코드를 활용한다. 다른 모델을 비교 학습한 결과는 아니다.

In [ ]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)

model= models.resnet50(weights='DEFAULT')
print(model.fc)

## 7. 기존 특징 추출층 동결 (Q3)
자동차도 자연 이미지이므로 사전학습된 시각적 특징을 활용한다. 제한된 데이터에서 기존 특징 추출층은 유지하고 마지막 분류층만 학습하는 전략이다. 데이터가 적다는 판단은 위 실제 이미지 수와 클래스 분포를 근거로 보고서에서 설명한다.

In [ ]:
for param in model.parameters():
    param.requires_grad= False

## 8. 마지막 FC층을 7개 클래스로 교체
기존 분류층을 교체하고 새 FC층만 학습 가능하도록 설정한다. 출력 번호는 ImageFolder의 class_names 순서와 같다.

In [ ]:
import torch.nn as nn

model.fc= nn.Linear(model.fc.in_features, len(class_names))
for param in model.fc.parameters():
    param.requires_grad= True

model= model.to(device)
print(model.fc)
total_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print('학습할 파라미터 수:', total_params)

## 9. 손실함수와 최적화
다중 분류이므로 CrossEntropyLoss를 사용하며, Adam으로 FC층만 최적화한다.

In [ ]:
import torch.optim as optim

criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.fc.parameters(), lr=0.001)

## 10. 모델 학습 및 Validation 확인
수업 1처럼 우선 3 Epoch 학습한다. model.eval()은 동결된 BatchNorm의 통계를 유지하며 FC층의 역전파를 막지 않는다. 손실은 수업 2처럼 배치 크기를 곱해 누적한 후 전체 이미지 수로 나눈다. Validation은 가중치를 갱신하지 않고 확인한다.

In [ ]:
epochs= 3

for epoch in range(epochs):
    model.eval()
    epoch_loss= 0
    epoch_accuracy= 0

    for images, labels in train_loader:
        images= images.to(device)
        labels= labels.to(device)

        optimizer.zero_grad()
        outputs= model(images)
        loss= criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * images.size(0)
        predictions= outputs.argmax(dim=1)
        epoch_accuracy += (predictions == labels).sum().item()

    val_loss= 0
    val_correct= 0
    with torch.no_grad():
        for images, labels in val_loader:
            images= images.to(device)
            labels= labels.to(device)
            outputs= model(images)
            loss= criterion(outputs, labels)
            predictions= outputs.argmax(dim=1)
            val_loss += loss.item() * images.size(0)
            val_correct += (predictions == labels).sum().item()

    print(f'Epoch {epoch + 1}/{epochs} | '
          f'Train Loss: {epoch_loss / len(train_dataset):.4f} | '
          f'Train Accuracy: {100 * epoch_accuracy / len(train_dataset):.2f}% | '
          f'Val Loss: {val_loss / len(val_dataset):.4f} | '
          f'Val Accuracy: {100 * val_correct / len(val_dataset):.2f}%')

## 11. Test 데이터로 최종 평가
수업 2처럼 학습에 사용하지 않은 Test에서 손실과 정확도를 계산하고, 정답과 예측을 저장한다. 여기서는 마지막 Epoch 모델을 평가한다.

In [ ]:
model.eval()
test_loss= 0
correct= 0
total= 0
y_true= []
y_pred= []

with torch.no_grad():
    for images, labels in test_loader:
        images= images.to(device)
        labels= labels.to(device)
        outputs= model(images)
        loss= criterion(outputs, labels)
        predictions= outputs.argmax(dim=1)

        test_loss += loss.item() * images.size(0)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

print(f'TEST LOSS: {test_loss / total:.4f}')
print(f'TEST ACCURACY: {100 * correct / total:.2f}%')

## 12. Precision / Recall / F1-Score와 혼동행렬
수업의 classification_report로 클래스별 성능을 확인한다. 추가한 혼동행렬은 행이 실제 클래스, 열이 예측 클래스이며 대각선 밖의 값이 오분류이다. Macro 평균과 Weighted 평균도 함께 확인한다.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(
    y_true, y_pred, target_names=class_names,
    labels=range(len(class_names)), digits=4, zero_division=0
))

# 오분류되는 클래스 조합을 확인하기 위한 그림
cm= confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
print('혼동행렬:')
print(cm)
fig, ax= plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap='Blues', values_format='d', xticks_rotation=45
)
plt.tight_layout()
plt.show()

## 13. 모델 가중치와 클래스명 저장
수업처럼 모델 객체 전체 대신 가중치와 클래스명 목록을 저장한다. 아래 파일은 이 셀을 다시 실행하면 덮어쓴다.

In [ ]:
model_path= dataset_path + '/car-resnet50-class-practice.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names
}, model_path)
print('모델 저장 완료:', model_path)

## 14. 저장된 모델 복원
수업 3처럼 같은 ResNet50 구조와 FC층을 만든 후 가중치를 복원한다. 클래스명은 저장된 순서를 사용한다. map_location은 CPU/GPU 환경 차이를 처리하고, weights_only=True는 가중치와 기본 자료형을 읽도록 명시한다.

In [ ]:
checkpoint= torch.load(model_path, map_location='cpu', weights_only=True)
class_names= checkpoint['class_names']
state_dict= checkpoint['model_state_dict']

model= models.resnet50(weights=None)
model.fc= nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(state_dict)
model= model.to(device)
model.eval()
print('모델 복원 완료:', class_names)

## 15. 새로운 자동차 사진 예측
수업과 같이 새 사진을 RGB로 읽고 동일하게 전처리한 뒤 배치 차원을 추가한다. 실행 시 Train/Validation/Test에 없는 사진의 Colab 또는 Drive 경로를 입력한다. 예측 라벨은 폴더명을 그대로 출력한다.

예측 신뢰도는 Softmax 점수이며 Test 전체의 Accuracy와 다르다. 실제 정답 확률을 보장하지 않고, 학습하지 않은 클래스도 기존 7개 중 하나로 예측될 수 있다.

In [ ]:
# input은 새 사진 경로를 직접 입력받기 위해 사용
image_path= input('새 자동차 사진 경로: ').strip()
image= Image.open(image_path).convert('RGB')
plt.imshow(image)
plt.axis('off')
plt.show()

image= transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs= model(image)
    probabilities= torch.softmax(outputs, dim=1)
    confidence, predicted= probabilities.max(dim=1)

print('회사명:', class_names[predicted.item()])
print(f'예측 신뢰도: {confidence.item() * 100:.2f}%')

## 제출 전 결과 확인 (Q3~Q4)
- Q3-2: Epoch별 Train/Validation 로그를 출력한 상태로 노트북을 저장한다.
- Q4-1: Test 지표와 혼동행렬에서 오분류가 많은 클래스 조합을 실제 값으로 설명한다.
- Q4-2: Train만 개선되고 Validation이 악화되는지 확인하여 과적합 가능성을 논의한다. 실행하지 않은 학습률 변경이나 증강은 시도했다고 작성하지 않는다.
- 무작위 분리는 클래스 비율을 정확히 맞추는 방식은 아니다. 학습에 사용하지 않은 Test로 평가하고, 원본 Train/Test에 중복 사진이 없는지도 확인한다.